# F06: Semantic Model Builder

Generate a Power BI `.bim` (TOM JSON) file from Spindle's retail schema, complete with tables, relationships, DAX measures, and M expressions.

## What You'll Learn
- Generate retail data and examine the schema metadata that drives semantic model export
- Use `SemanticModelExporter` to create a `.bim` file
- Understand the TOM JSON structure: tables, columns, relationships, measures, partitions
- See how M expressions differ for `lakehouse`, `warehouse`, and `sql_database` source types
- Use the `export-model` CLI command for quick exports

## Prerequisites
- Python 3.10+
- `sqllocks-spindle` installed
- For deployment: a Fabric workspace with XMLA endpoint access

## Time Estimate
~20 minutes

In [ ]:
# %pip install sqllocks-spindle

## Step 1: Generate Retail Data

**What we're about to do:** Generate a retail dataset and inspect the schema metadata. The `SemanticModelExporter` reads table definitions, column types, and FK relationships from the schema to build the TOM model.

**Why this matters:** The semantic model isn't generated from the DataFrames -- it's generated from the *schema*. Spindle schemas carry enough metadata (types, nullability, FK relationships) to fully define a Power BI model without manual configuration.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="small", seed=42)

# Inspect schema metadata that drives the semantic model
schema = result.schema
print(f"Domain: {schema.model.domain}")
print(f"Tables: {list(schema.tables.keys())}")
print(f"Relationships: {len(schema.relationships)}")
print()

# Show relationships (these become the semantic model's relationship definitions)
for rel in schema.relationships:
    print(f"  {rel.child} -> {rel.parent}  ({rel.child_columns} -> {rel.parent_columns})")

## Step 2: Export the .bim File

**What we're about to do:** Use `SemanticModelExporter` to generate a `.bim` file. This is a JSON file following the TOM (Tabular Object Model) specification at compatibility level 1604 -- the format used by Power BI Desktop, XMLA endpoints, and Fabric semantic models.

**Key concept:** The `source_type` parameter controls what M (Power Query) expressions are generated for table partitions:
- `"lakehouse"` -- `Lakehouse.Contents(null)` expressions with workspace/item IDs
- `"warehouse"` -- `Sql.Database()` expressions with server/database references
- `"sql_database"` -- Same as warehouse (both use `Sql.Database()`)

In [ ]:
from sqllocks_spindle.fabric.semantic_model_writer import SemanticModelExporter

exporter = SemanticModelExporter()

# Export a .bim file for a lakehouse source
bim_path = exporter.export_bim(
    schema=schema,
    source_type="lakehouse",
    source_name="RetailLakehouse",
    output_path="retail_model.bim",
    include_measures=True,
)

print(f"Exported .bim file: {bim_path}")
print(f"File size: {bim_path.stat().st_size / 1024:.1f} KB")

## Step 3: Inspect the TOM JSON Structure

**What we're about to do:** Load the `.bim` file and inspect the key sections: tables (with columns), relationships (from FK definitions), and measures (auto-generated DAX).

**What to expect:**
- Each table has typed columns with `sourceColumn` references
- PK columns are marked `isKey: true`
- FK columns are hidden by default (join keys aren't interesting to end users)
- Relationships use `oneDirection` cross-filtering
- Measures include `COUNTROWS`, `SUM`, and `AVERAGE` for numeric columns

In [ ]:
import json

with open("retail_model.bim") as f:
    bim = json.load(f)

# Top-level structure
print(f"Model name: {bim['name']}")
print(f"Compatibility level: {bim['compatibilityLevel']}")
print(f"Tables: {len(bim['model']['tables'])}")
print(f"Relationships: {len(bim['model']['relationships'])}")

# Show tables with column counts and measure counts
print(f"\n{'Table':<25} {'Columns':>8} {'Measures':>10}")
print("-" * 45)
for table in bim["model"]["tables"]:
    cols = len(table["columns"])
    measures = len(table.get("measures", []))
    print(f"{table['name']:<25} {cols:>8} {measures:>10}")

# Show relationships
print("\nRelationships:")
for rel in bim["model"]["relationships"]:
    print(f"  {rel['fromTable']}.{rel['fromColumn']} -> {rel['toTable']}.{rel['toColumn']}")

In [ ]:
# Show auto-generated DAX measures for the first table that has them
for table in bim["model"]["tables"]:
    measures = table.get("measures", [])
    if measures:
        print(f"Auto-generated DAX measures for '{table['name']}':")
        for m in measures[:6]:  # Show first 6 measures
            print(f"  {m['name']:<30} = {m['expression']}")
        if len(measures) > 6:
            print(f"  ... and {len(measures) - 6} more")
        break

# Show the M expression for one table partition
first_table = bim["model"]["tables"][0]
partition = first_table["partitions"][0]
print(f"\nM expression for '{first_table['name']}' (lakehouse source):")
print(partition["source"]["expression"])

## Step 4: Compare M Expressions Across Source Types

**What we're about to do:** Generate the same semantic model with different `source_type` values to see how the M expressions change. This determines how Power BI connects to your data.

**Key difference:**
- **Lakehouse** uses `Lakehouse.Contents(null)` -- connects via workspace/item GUIDs
- **Warehouse** uses `Sql.Database("server", "database")` -- connects via SQL endpoint
- **SQL Database** uses the same `Sql.Database()` pattern as Warehouse

In [ ]:
# Compare M expressions for each source type
sample_table = list(schema.tables.keys())[0]

for source_type in ["lakehouse", "warehouse", "sql_database"]:
    tom = exporter.to_dict(
        schema=schema,
        source_type=source_type,
        source_name="MyRetailSource",
        schema_name="dbo",
    )
    # Find the first table's partition M expression
    first_tbl = tom["model"]["tables"][0]
    m_expr = first_tbl["partitions"][0]["source"]["expression"]
    print(f"--- source_type='{source_type}' ---")
    print(m_expr)
    print()

# CLI equivalent (for quick exports without writing Python)
print("CLI equivalent:")
print("  spindle export-model --domain retail --source-type lakehouse --output retail_model.bim")

## What's Next?

You've generated a complete `.bim` semantic model with tables, relationships, DAX measures, and M expressions for all three Fabric source types.

- **F02: Warehouse Dimensional Load** -- Load the retail data to a Fabric Warehouse, then deploy this `.bim` on top of it for a complete analytics stack
- **F01: Medallion Architecture** -- Build a Bronze/Silver/Gold pipeline and export a semantic model from the gold tier
- **T06: Star Schema Export** -- Learn how `StarSchemaTransform` creates the dim/fact layout that pairs perfectly with semantic models